### we are using pagIndex to implement vectorLess Rag

**Vectorless RAG** = no embeddings, no vector DB, no chunking.

1. **Index** – PageIndex turns the PDF into a *tree* (like a smart table of contents): each node has a title, page range and summary.
2. **Tree search** – the LLM *reads the tree* and reasons about which sections are likely to contain the answer (like a human skimming a ToC).
3. **Retrieve** – we pull the full text of only those nodes.
4. **Generate** – the LLM answers using that text, citing the sections/pages.

In [8]:
import os
import json
import time
from dotenv import load_dotenv

load_dotenv()  # loads GROQ_API_KEY and PAGEINDEX_API_KEY from .env

True

In [ ]:
from pageindex import PageIndexClient
from groq import Groq

pi_client = PageIndexClient(api_key=os.environ["PAGEINDEX_API_KEY"])
groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])

# See which models your key can use: [m.id for m in groq_client.models.list().data]
MODEL = "openai/gpt-oss-120b"

#### Step 1: Get a PDF and index it with PageIndex
Downloads the *Attention Is All You Need* paper as a sample. Swap `PDF_PATH` for any PDF of your own.

In [10]:
import requests

PDF_PATH = "attention_is_all_you_need.pdf"
PDF_URL = "https://arxiv.org/pdf/1706.03762"

if not os.path.exists(PDF_PATH):
    resp = requests.get(PDF_URL, timeout=60)
    resp.raise_for_status()
    with open(PDF_PATH, "wb") as f:
        f.write(resp.content)
    print("Downloaded", PDF_PATH)

Downloaded attention_is_all_you_need.pdf


In [11]:
# Reuse the document if it was already uploaded, otherwise submit it and wait for indexing
try:
    doc_id = pi_client.get_document_id(os.path.basename(PDF_PATH))
    print("Found existing document:", doc_id)
except Exception:
    doc_id = pi_client.submit_document(PDF_PATH, wait=True)["doc_id"]
    print("Indexed new document:", doc_id)

while not pi_client.is_retrieval_ready(doc_id):
    print("Waiting for tree to be ready...")
    time.sleep(5)
print("Ready!")

Indexed new document: pi-cmulof7b7004t0cnypa3yeah6
Ready!


#### Step 2: Look at the tree
This is the "index" — no vectors, just a hierarchy of sections with summaries.

In [12]:
# Full tree (with text) – used later to fetch node content
tree = pi_client.get_tree(doc_id, node_summary=True)["result"]

def flatten(nodes):
    """node_id -> node, for every node in the tree"""
    out = {}
    for n in nodes:
        out[n["node_id"]] = n
        out.update(flatten(n.get("nodes", [])))
    return out

node_map = flatten(tree)

def compact_tree(nodes):
    """Tree without the full text – this is what the LLM reasons over"""
    return [
        {
            "node_id": n["node_id"],
            "title": n["title"],
            "page": n.get("page_index"),
            "summary": n.get("summary") or n.get("prefix_summary", ""),
            "nodes": compact_tree(n.get("nodes", [])),
        }
        for n in nodes
    ]

tree_for_llm = compact_tree(tree)

def print_tree(nodes, depth=0):
    for n in nodes:
        print("  " * depth + f"[{n['node_id']}] {n['title']} (p.{n['page']})")
        print_tree(n["nodes"], depth + 1)

print_tree(tree_for_llm)

[0000] Attention Is All You Need (p.1)
  [0001] Abstract (p.1)
  [0002] 1 Introduction (p.2)
  [0003] 2 Background (p.2)
  [0004] 3 Model Architecture (p.2)
    [0005] 3.1 Encoder and Decoder Stacks (p.3)
    [0006] 3.2 Attention (p.3)
      [0007] 3.2.1 Scaled Dot-Product Attention (p.4)
      [0008] 3.2.2 Multi-Head Attention (p.4)
      [0009] 3.2.3 Applications of Attention in our Model (p.5)
    [0010] 3.3 Position-wise Feed-Forward Networks (p.5)
    [0011] 3.4 Embeddings and Softmax (p.5)
    [0012] 3.5 Positional Encoding (p.6)
  [0013] 4 Why Self-Attention (p.6)
  [0014] 5 Training (p.7)
  [0015] 6 Results (p.8)
    [0016] 6.1 Machine Translation (p.8)
    [0017] 6.2 Model Variations (p.8)
    [0018] 6.3 English Constituency Parsing (p.9)
  [0019] 7 Conclusion (p.10)
  [0020] References (p.10)
  [0021] Attention Visualizations (p.13)


#### Step 3: Tree search – the LLM picks the relevant nodes by reasoning

In [13]:
def tree_search(query: str) -> dict:
    prompt = f"""You are given a question and the tree structure of a document.
Each node has a node_id, title, page number and summary.
Find all nodes that are likely to contain the answer to the question.

Question: {query}

Document tree:
{json.dumps(tree_for_llm, indent=1)}

Reply in JSON exactly like this:
{{"thinking": "<your reasoning about which nodes are relevant>", "node_list": ["node_id_1", "node_id_2"]}}"""

    response = groq_client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
        temperature=0,
    )
    return json.loads(response.choices[0].message.content)

#### Step 4 + 5: Retrieve node text and generate the answer

In [14]:
def generate_answer(query: str, node_ids: list[str]) -> str:
    context = "\n\n".join(
        f"### {node_map[nid]['title']} (page {node_map[nid].get('page_index')})\n{node_map[nid].get('text', '')}"
        for nid in node_ids
        if nid in node_map
    )
    prompt = f"""Answer the question using only the context below.
Cite the section titles / page numbers you used. If the context doesn't contain the answer, say so.

Question: {query}

Context:
{context}"""

    response = groq_client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return response.choices[0].message.content


def vectorless_rag(query: str) -> str:
    search = tree_search(query)
    print("🧠 Reasoning:", search["thinking"])
    print("📄 Retrieved nodes:")
    for nid in search["node_list"]:
        if nid in node_map:
            print(f"   [{nid}] {node_map[nid]['title']} (p.{node_map[nid].get('page_index')})")
    print()
    return generate_answer(query, search["node_list"])

In [15]:
answer = vectorless_rag("How does multi-head attention work and why is it useful?")
print(answer)

NotFoundError: Error code: 404 - {'error': {'message': 'The model `llama-3.3-70b-versatile` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}

In [ ]:
answer = vectorless_rag("What BLEU score did the Transformer get on English-to-German translation?")
print(answer)